# Part 1: Data Cleaning - Delhi Housing Dataset

In [ ]:
import pandas as pd
import numpy as np

# 1. Load the dataset verbatim
df = pd.read_csv('Delhi.csv')

# 2. Handle Placeholders
# Replacing '9' (no information) with '0' (not available) for binary columns
df.replace(9, 0, inplace=True)

print(f"Initial data loaded. Shape: {df.shape}")

In [ ]:
# List of columns highlighted in yellow for consolidation
highlighted_cols = [
    'MaintenanceStaff', 'Gymnasium', 'SwimmingPool', 'JoggingTrack', 
    'RainWaterHarvesting', 'ShoppingMall', 'ATM', 'School', 
    'StaffQuarter', 'Cafeteria', 'MultipurposeRoom', 'Hospital', 
    'WashingMachine', 'BED', 'VaastuCompliant', 'Microwave', 
    'GolfCourse', 'TV', 'DiningTable', 'Sofa', 'Wardrobe', 'Refrigerator'
]

# Create the Total_Amenities column
df['Total_Amenities'] = df[highlighted_cols].sum(axis=1)

# Defining columns to keep as independent features (unhighlighted in image)
white_cols = [
    'Price', 'Area', 'Location', 'No. of Bedrooms', 'Resale', 
    'LandscapedGardens', 'IndoorGames', 'Intercom', 'SportsFacility', 
    'ClubHouse', '24X7Security', 'PowerBackup', 'CarParking', 
    'Gasconnection', 'AC', 'Wifi', "Children'splayarea", 'LiftAvailable'
]

# Construct final dataframe
df_cleaned = df[white_cols + ['Total_Amenities']]

# Save the resulting dataframe to a new CSV file
df_cleaned.to_csv('delhi_cleaned.csv', index=False)
print("Cleaned data successfully saved to 'delhi_cleaned.csv'.")

# Part 2: Multiple Linear Regression

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Load the cleaned dataset verbatim
ml_data = pd.read_csv('delhi_cleaned.csv')

# 2. Data Preprocessing
# The 'Location' column is categorical text data. We need to convert it into 
# numerical format using One-Hot Encoding so the ML model can understand it.
ml_data = pd.get_dummies(ml_data, columns=['Location'], drop_first=True)

# 3. Define Features (X) and Target (y)
# We want to predict 'Price', so it becomes our target variable (y).
X = ml_data.drop('Price', axis=1)
y = ml_data['Price']

# 4. Train-Test Split
# Split the dataset into 80% training data and 20% testing data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# 5. Initialize and Train the Model
model = LinearRegression()
model.fit(X_train, y_train)
print("Model training complete.")

# 6. Make Predictions
y_pred = model.predict(X_test)

# 7. Evaluate the Model
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("\n--- Model Evaluation Metrics ---")
print(f"Mean Absolute Error (MAE): {mae:,.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:,.2f}")
print(f"R-squared (R2) Score: {r2:.4f}")

In [ ]:
import joblib

# 1. Save the trained model
joblib.dump(model, 'delhi_housing_model.joblib')

# 2. Save the exact list of feature columns the model was trained on
# This is CRITICAL for the Streamlit app to work properly
joblib.dump(list(X.columns), 'model_columns.joblib')

print("Model and columns saved successfully! Ready to hand off.")